[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/ia-local/04-transformers-js-python.ipynb)

# Transformers.js y Alternativas Python

> Compara Transformers.js (modelos en el navegador con JavaScript) con su equivalente
> Python local. Entiende cuándo usar cada opción y cómo integrarlos en proyectos web.

## Objetivos
- Entender qué es Transformers.js y sus casos de uso web
- Replicar en Python los mismos pipelines disponibles en Transformers.js
- Comparar rendimiento y casos de uso de cada entorno
- Construir un pipeline de clasificación offline en Python

## 1. Instalación de dependencias

In [ ]:
%pip install transformers torch sentencepiece --quiet

## 2. ¿Qué es Transformers.js?

Transformers.js es la librería de HuggingFace para ejecutar modelos directamente en el navegador
o en Node.js, sin servidor ni API. Usa WebAssembly y WebGPU para acelerar la inferencia.

```javascript
// Ejemplo Transformers.js en Node.js
import { pipeline } from '@xenova/transformers';

// Análisis de sentimiento en el navegador
const clasificador = await pipeline('sentiment-analysis');
const resultado = await clasificador('Este producto es increíble!');
console.log(resultado); // [{ label: 'POSITIVE', score: 0.998 }]

// Embeddings sin servidor
const extractor = await pipeline('feature-extraction', 'Xenova/all-MiniLM-L6-v2');
const embedding = await extractor('Hola mundo', { pooling: 'mean', normalize: true });
```

**Cuándo usar Transformers.js:**
- Apps web que necesitan privacidad total (datos nunca salen del navegador)
- Aplicaciones offline o con conectividad limitada
- Extensiones de navegador con procesamiento de texto
- Demos interactivas sin infraestructura de servidor

## 3. Equivalente Python: análisis de sentimiento

In [ ]:
from transformers import pipeline
import time

# Pipeline de sentimiento — equivalente a Transformers.js 'sentiment-analysis'
print("Cargando clasificador de sentimiento...")
clasificador = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

textos = [
    "This product is absolutely amazing, I love it!",
    "Terrible experience, totally disappointed.",
    "It's okay, nothing special but does the job.",
    "Best purchase I've made this year!",
    "Would not recommend to anyone."
]

print("\n=== ANÁLISIS DE SENTIMIENTO (Python local) ===")
inicio = time.perf_counter()
resultados = clasificador(textos)
duracion = time.perf_counter() - inicio

for texto, resultado in zip(textos, resultados):
    emoji = "😊" if resultado['label'] == 'POSITIVE' else "😞"
    print(f"  {emoji} '{texto[:50]}' → {resultado['label']} ({resultado['score']:.1%})")

print(f"\nTiempo total: {duracion:.3f}s | Media: {duracion/len(textos):.3f}s por texto")

## 4. Equivalente Python: extracción de embeddings

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

# Equivalente exacto del modelo MiniLM de Transformers.js
print("Cargando MiniLM para embeddings...")
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
modelo = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def obtener_embedding(texto: str) -> np.ndarray:
    """Genera embedding normalizado — equivalente a Transformers.js feature-extraction."""
    tokens = tokenizer(texto, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        salida = modelo(**tokens)
    # Mean pooling sobre los token embeddings
    mascara = tokens['attention_mask'].unsqueeze(-1).float()
    embedding = (salida.last_hidden_state * mascara).sum(1) / mascara.sum(1)
    # Normalizar
    embedding = torch.nn.functional.normalize(embedding, p=2, dim=1)
    return embedding.numpy()[0]

frases = [
    "IA en el navegador sin servidor",
    "Machine learning offline en el cliente",
    "Receta de pasta carbonara"
]

embeddings = [obtener_embedding(f) for f in frases]
print(f"Dimensión de cada embedding: {embeddings[0].shape[0]}")

print("\n=== SIMILITUD COSENO ENTRE FRASES ===")
for i in range(len(frases)):
    for j in range(i + 1, len(frases)):
        sim = float(np.dot(embeddings[i], embeddings[j]))
        print(f"  [{i+1}↔{j+1}] Similitud: {sim:.3f} | '{frases[i]}' vs '{frases[j]}'")

## 5. Tabla comparativa Python vs Transformers.js

In [ ]:
import pandas as pd

comparativa = pd.DataFrame([
    {"Criterio": "Entorno", "Python local": "Scripts, notebooks, backend", "Transformers.js": "Navegador, Node.js, extensiones"},
    {"Criterio": "Instalación", "Python local": "pip install transformers torch", "Transformers.js": "npm install @xenova/transformers"},
    {"Criterio": "Rendimiento", "Python local": "Mejor en CPU/GPU dedicada", "Transformers.js": "WebAssembly/WebGPU, más lento"},
    {"Criterio": "Privacidad", "Python local": "Total (datos en máquina)", "Transformers.js": "Total (datos en navegador)"},
    {"Criterio": "Modelos disponibles", "Python local": "Todos los de HuggingFace", "Transformers.js": "Subconjunto optimizado (ONNX)"},
    {"Criterio": "Formato de modelos", "Python local": "PyTorch / SafeTensors", "Transformers.js": "ONNX (conversión necesaria)"},
    {"Criterio": "Mejor para", "Python local": "Análisis datos, backend, MLOps", "Transformers.js": "Apps web offline, demos, extensiones"},
    {"Criterio": "GPU requerida", "Python local": "Opcional (mejora velocidad)", "Transformers.js": "No (WebGPU opcional)"},
])

print("=== COMPARATIVA PYTHON LOCAL vs TRANSFORMERS.JS ===")
print(comparativa.to_string(index=False))

print("\n=== GUÍA DE DECISIÓN ===")
print("¿Necesitas integrarlo en una web/extensión de navegador? → Transformers.js")
print("¿Estás en un entorno Python (script, backend, notebook)? → transformers + PyTorch")
print("¿Necesitas los modelos más grandes y precisos? → Python local o API cloud")
print("¿Quieres prototipo rápido sin backend? → Transformers.js con demo en navegador")